In [ ]:
#---------------
# IMPORTALAS
#---------------

import os
import warnings  # figyelmeztetések elnémítása futtatáskor
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import yfinance as yf
from scipy import stats
from scipy.stats import jarque_bera, norm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from arch import arch_model
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from statsmodels.stats.multitest import multipletests
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
warnings.filterwarnings('ignore')
torch.manual_seed(42)  # reprodukálhatóság: PyTorch véletlenszám-generátor rögzítése
np.random.seed(42)  # reprodukálhatóság: NumPy véletlenszám-generátor rögzítése
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # GPU-t használ ha elérhető, különben CPU
print(f'PyTorch: {torch.__version__} | Device: {DEVICE}')

In [ ]:
# =============================================================================
# KONFIGURÁCIÓ – v14
# =============================================================================

TICKER     = '^GSPC'  # S&P 500 index Yahoo Finance szimbóluma
START_DATE = '2013-01-01'
END_DATE   = '2025-01-01'

ROLLING_WINDOWS = [1000]  # csak 1000 napos ablak marad; a 252 napos LSTM-hez elégtelen

LAMBDA_EWMA    = 0.94  # RiskMetrics (1996) ajánlása: λ=0.94 napi adathoz
REFIT_STEP     = 21  # 21 kereskedési nap ≈ 1 hónap; minden modell havonta frissül
RV_WINDOW      = 21  # 21 napos rolling ablak a Garman-Klass RV kiszámításához
TDPY           = 252

# LSTM – v8 alap konfiguráció
LSTM_LAYERS    = 2  # kétrétegű LSTM – Fischer & Krauss (2018) alapján
LSTM_UNITS     = 64  # 64 neuron/réteg; a v8-ban validált optimum
LSTM_DROPOUT   = 0.1  # kis dropout a túlillesztés ellen kétrétegű hálóban
LSTM_EPOCHS    = 50  # maximum epochszám; korai leállás (patience) rövidítheti
LSTM_PATIENCE  = 5  # 5 epoch javulás nélkül → korai leállás
LSTM_BATCH     = 32
LSTM_LOOKBACK  = 60  # 60 kereskedési nap ≈ 3 hónap múltbeli ablak
LSTM_REFIT     = 21      # egységes minden modellnél (REFIT_STEP=21)  # LSTM is 21 naponta újratanul, konzisztens a klasszikus modellekkel

VAR_LEVELS     = [0.95, 0.99]  # VaR backtesting két konfidencia-szinten

FIG_DIR  = 'figures'
DATA_DIR = 'data'
os.makedirs(FIG_DIR,  exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

TICKER_NAMES = {
    '^GSPC': 'S&P 500', '^OEX': 'S&P 100', '^DJI': 'Dow Jones',
    '^IXIC': 'NASDAQ',  '^FTSE': 'FTSE 100', '^GDAXI': 'DAX',
}
DISPLAY_NAME = TICKER_NAMES.get(TICKER, TICKER)
safe_ticker  = TICKER.replace('^', '')

PERIODS = {
    1: ('2019-01-01', '2020-01-31', 'Pre-COVID (2019)'),
    2: ('2020-02-01', '2020-12-31', 'COVID-sokk (2020)'),
    3: ('2022-01-01', '2022-12-31', 'Kamatemelés (2022)'),
}
PERIOD_COLORS = {1: '#27ae60', 2: '#e74c3c', 3: '#e67e22'}

print(f'Ticker:          {TICKER} ({DISPLAY_NAME})')
print(f'Letöltési idő:   {START_DATE} – {END_DATE}')
print(f'Rolling ablak:   {ROLLING_WINDOWS}')
print(f'LSTM alap:       {LSTM_LAYERS} réteg, {LSTM_UNITS} neuron, lookback={LSTM_LOOKBACK}, refit={LSTM_REFIT} nap')
print(f'Modellek:        EWMA · GARCH · GJR-GARCH · HAR-RV · LSTM · LSTM-GARCH hibrid')
print(f'VaR szintek:     {VAR_LEVELS}')


In [ ]:
# =============================================================================
# GARMAN-KLASS RV + HAR KOMPONENSEK – v11
# =============================================================================
# GK_RV      – 21 napos rolling → benchmark
# GK_daily   – napi, rolling nélkül → LSTM target + HAR-RV regresszor
#
# HAR-RV komponensek (Corsi 2009) – mind shift(1)-gyel, leakage-mentes:
#   log_GK_d  = log(GK_daily.shift(1))               – napi RV
#   log_GK_w  = log(GK_daily.shift(1).rolling(5).mean())  – heti RV
#   log_GK_m  = log(GK_daily.shift(1).rolling(22).mean()) – havi RV
# =============================================================================

def compute_garman_klass_rv(df, window=21, tdpy=252):
    log_hl = np.log(df['High']  / df['Low'])  # napi magas-alacsony log-arány (Garman-Klass képlet első tagja)
    log_co = np.log(df['Close'] / df['Open'])  # napi záró-nyitó log-arány (Garman-Klass képlet második tagja)
    gk_sq  = 0.5 * log_hl**2 - (2 * np.log(2) - 1) * log_co**2  # Garman-Klass (1980) variance becslő: σ²≈0.5·(H/L)²-(2ln2-1)·(C/O)²
    return np.sqrt(gk_sq.rolling(window, min_periods=window//2).mean().clip(lower=0) * tdpy)  # rolling átlag → simított, évesített GK volatilitás (σ·√252)

def compute_garman_klass_daily(df, tdpy=252):
    log_hl = np.log(df['High']  / df['Low'])  # napi magas-alacsony log-arány (Garman-Klass képlet első tagja)
    log_co = np.log(df['Close'] / df['Open'])  # napi záró-nyitó log-arány (Garman-Klass képlet második tagja)
    gk_sq  = 0.5 * log_hl**2 - (2 * np.log(2) - 1) * log_co**2  # Garman-Klass (1980) variance becslő: σ²≈0.5·(H/L)²-(2ln2-1)·(C/O)²
    return np.sqrt(gk_sq.clip(lower=0) * tdpy)

def compute_all_rv_and_features(df, window=21):
    df = df.copy()
    df['log_return']   = np.log(df['Close'] / df['Close'].shift(1))  # log-hozam: ln(P_t / P_{t-1}) – normális eloszláshoz közelebb mint egyszerű hozam
    df['simple_return'] = df['Close'].pct_change()
    df['GK_RV']        = compute_garman_klass_rv(df, window=window)  # 21 napos simított GK RV → ez lesz a benchmark (ground truth)
    df['GK_daily']     = compute_garman_klass_daily(df)  # simítatlan napi GK volatilitás → LSTM target és HAR-RV regresszor
    # HAR-RV komponensek – shift(1): aznapi GK_daily nem kerül be
    eps               = 1e-8
    gk_lag            = df['GK_daily'].shift(1)  # shift(1): leakage-mentes, az aznapi adat nem kerül be a featurekbe
    df['log_GK_d']    = np.log(gk_lag.clip(lower=eps))  # napi HAR-RV komponens (log-skála): tegnapi volatilitás
    df['log_GK_w']    = np.log(gk_lag.rolling(5).mean().clip(lower=eps))  # heti HAR-RV komponens: 5 napos visszatekintő átlag
    df['log_GK_m']    = np.log(gk_lag.rolling(22).mean().clip(lower=eps))  # havi HAR-RV komponens: 22 napos visszatekintő átlag
    return df

# =============================================================================
# ADATOK BETÖLTÉSE
# =============================================================================
file_path = f'{DATA_DIR}/{safe_ticker}_prices_{START_DATE}_{END_DATE}.csv'
feat_check = ['GK_RV', 'GK_daily', 'log_GK_d', 'log_GK_w', 'log_GK_m']

if os.path.exists(file_path):  # cache: ha már letöltöttük, ne töltsük le újra
    prices_data = pd.read_csv(file_path, index_col=0, parse_dates=True)
    print(f'✓ Betöltve: {file_path} ({len(prices_data)} nap)')
    if not all(c in prices_data.columns for c in feat_check):
        prices_data = compute_all_rv_and_features(prices_data)
        prices_data.to_csv(file_path)
        print('  Feature-ök újraszámítva.')
else:
    print(f'Letöltés: {TICKER}...')
    raw = yf.Ticker(TICKER).history(start=START_DATE, end=END_DATE)
    if raw.empty:
        raise ValueError(f'Nem sikerült adatot letölteni: {TICKER}')
    prices_data = compute_all_rv_and_features(raw)
    prices_data.to_csv(file_path)
    print(f'✓ Letöltve és mentve: {len(prices_data)} nap')

prices_data.index = pd.to_datetime(prices_data.index, utc=True).tz_localize(None)  # időzóna-naivvá alakítás: tz-aware index okozhat merge-hibát
prices_data       = prices_data.dropna(subset=['log_return'])
prices_data_full  = prices_data.copy()
log_returns_full  = prices_data_full['log_return']
gk_rv_full        = prices_data_full['GK_RV']
gk_daily_full     = prices_data_full['GK_daily']

print(f'Adatsor: {prices_data.index[0].date()} – {prices_data.index[-1].date()} '
      f'({len(prices_data)} nap)')
print(f'GK_RV   (rolling 21): {gk_rv_full.dropna().shape[0]} érvényes nap')
print(f'GK_daily (napi):      {gk_daily_full.dropna().shape[0]} érvényes nap')
for f in ['log_GK_d', 'log_GK_w', 'log_GK_m']:
    print(f'  {f}: {prices_data[f].notna().sum()} érvényes nap')


# =============================================================================
# PERIÓDUSOK KIVÁGÁSA
# =============================================================================
period_data = {}
for pid, (start, end, name) in PERIODS.items():
    mask = (prices_data.index >= start) & (prices_data.index <= end)
    df   = prices_data[mask].dropna(subset=['log_return'])
    period_data[pid] = dict(
        df       = df,
        name     = name,
        start    = start,
        end      = end,
        prices   = df['Close'],
        log_ret  = df['log_return'],
        gk_rv    = df['GK_RV'],
        gk_daily = df['GK_daily'],
    )
    ann_vol = df['log_return'].std() * np.sqrt(TDPY) * 100
    print(f'  Periódus {pid}: {name:28s} | {start} – {end} | '
          f'{len(df):4d} nap | vol={ann_vol:.1f}%')


In [ ]:
# 3.1 Árfolyamok – 3×1 egymás alá (TDK formátum)
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=False)  # 3 periódus egymás alatt, közös x-tengely kikapcsolva (különböző időszakok)
for i, (pid, pd_) in enumerate(period_data.items()):
    ax = axes[i]
    ax.plot(pd_['prices'], color=PERIOD_COLORS[pid], linewidth=1.2)  # záróárfolyam idősor az adott periódus színével
    ax.set_title(pd_['name'], fontsize=11, fontweight='bold')
    ax.set_ylabel('Záróárfolyam (USD)', fontsize=9)
    ax.tick_params(axis='x', rotation=20)
    ax.grid(True, alpha=0.3)
    if i == 2:
        ax.set_xlabel('Dátum', fontsize=9)
plt.suptitle(f'{DISPLAY_NAME} – Záróárfolyamok periódusokként',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/01_arfolyamok_{safe_ticker}.png', dpi=150, bbox_inches='tight')  
plt.show()
print(f'✓ Mentve: {FIG_DIR}/01_arfolyamok_{safe_ticker}.png')


In [ ]:
# 3.2 + 3.3 Log-hozamok és eloszlások – egy ábrán (2×3 grid, TDK formátum)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

print('\nLeíró statisztikák:')
print(f'{"Periódus":<28} {"Mean":>8} {"Std":>8} {"Skew":>8} {"Kurt":>8} {"JB p":>10}')
print('-' * 72)

for i, (pid, pd_) in enumerate(period_data.items()):
    r   = pd_['log_ret'].values * 100
    idx = pd_['log_ret'].index
    mu, sigma = r.mean(), r.std()
    jb_stat, jb_p = jarque_bera(r)  # Jarque-Bera teszt: H₀: normális eloszlás – p<0.05 → elveti
    skew = stats.skew(r); kurt = stats.kurtosis(r)  # ferdeség: negatív → bal oldali vastag farok (nagy esések gyakoribbak)  # csúcsosság (excess): >0 → leptokurtikus, vastag farkú eloszlás

    # Felső sor: hozamok
    ax_ret = axes[0, i]
    ax_ret.plot(idx, r, color=PERIOD_COLORS[pid], linewidth=0.6, alpha=0.85)
    ax_ret.axhline(0, color='black', linewidth=0.5, linestyle='--')  # vízszintes nulla-vonal referenciának
    ax_ret.set_title(pd_['name'], fontsize=11, fontweight='bold')
    ax_ret.set_ylabel('Log-hozam (%)' if i == 0 else '', fontsize=9)
    ax_ret.tick_params(axis='x', rotation=20)
    ax_ret.grid(True, alpha=0.3)

    # Alsó sor: eloszlások
    ax_dist = axes[1, i]
    ax_dist.hist(r, bins=40, density=True, alpha=0.55,  # empirikus eloszlás hisztogramja
                 color=PERIOD_COLORS[pid], label='Empirikus')
    x = np.linspace(r.min(), r.max(), 200)
    ax_dist.plot(x, norm.pdf(x, mu, sigma), 'k-',  lw=1.5, label='Normál')
    df_t, loc_t, scale_t = stats.t.fit(r)  # Student-t illesztés MLE-vel: ν szabadságfok becslése adatból
    ax_dist.plot(x, stats.t.pdf(x, df_t, loc_t, scale_t),
                 'b--', lw=1.5, label=f'Student-t (ν={df_t:.1f})')
    ax_dist.set_xlabel('Log-hozam (%)', fontsize=9)
    ax_dist.set_ylabel('Sűrűség' if i == 0 else '', fontsize=9)
    ax_dist.legend(fontsize=8)
    ax_dist.grid(True, alpha=0.3)

    print(f'{pd_["name"]:<28} {mu:8.4f} {sigma:8.4f} {skew:8.4f} {kurt:8.4f} {jb_p:10.4f}')

plt.suptitle(f'{DISPLAY_NAME} – Log-hozamok és eloszlások periódusokként',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/02_hozamok_eloszlasok_{safe_ticker}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✓ Mentve: {FIG_DIR}/02_hozamok_eloszlasok_{safe_ticker}.png')


In [ ]:
# 3.4 ACF/PACF – hozamok + négyzetes hozamok
for pid, pd_ in period_data.items():
    r    = pd_['log_ret']
    conf = 1.96 / np.sqrt(len(r))  # 95%-os konfidencia-sáv határa az ACF-ábrán (Bartlett-képlet)
    fig, axes = plt.subplots(2, 2, figsize=(13, 7))
    plot_acf( r,    ax=axes[0,0], lags=30, alpha=0.05, title='ACF – hozamok')  # nyers hozamok autokorrelációja: ARMA struktúra vizsgálata
    plot_pacf(r,    ax=axes[0,1], lags=30, alpha=0.05, title='PACF – hozamok')
    plot_acf( r**2, ax=axes[1,0], lags=30, alpha=0.05, title='ACF – négyzetes hozamok')  # négyzetes hozamok ACF-je: volatilitás-klaszterezés kimutatása
    plot_acf( r.abs(), ax=axes[1,1], lags=30, alpha=0.05, title='ACF – abszolút hozamok')  # abszolút hozamok ACF-je: robusztusabb volatilitás-proxy
    for ax in axes.flatten():
        ax.grid(True, alpha=0.3)
    plt.suptitle(f'{DISPLAY_NAME} – {pd_["name"]} – ACF/PACF', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/04_acf_P{pid}_{safe_ticker}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Mentve: {FIG_DIR}/04_acf_P{pid}_{safe_ticker}.png')

In [ ]:
# =============================================================================
# HIBAMÉRŐK
# =============================================================================

def calc_metrics(y_true, y_pred, name):
    """RMSE, MAE, Q-LIKE – Patton (2011)"""
    mask  = ~(np.isnan(y_true) | np.isnan(y_pred))  # NaN-ok kizárása mindkét sorozatból egyszerre
    yt, yp = np.array(y_true)[mask], np.array(y_pred)[mask]
    if len(yt) == 0:
        return {'Modell': name, 'Q-LIKE': np.nan, 'RMSE': np.nan, 'MAE': np.nan, 'N': 0}
    rmse  = np.sqrt(mean_squared_error(yt, yp))  # RMSE: outlierekre érzékeny, nagy hibákat aránytalanul bünteti
    mae   = mean_absolute_error(yt, yp)  # MAE: robusztus, egyforma súlyú abszolút eltérések
    eps   = 1e-10  # numerikus stabilitás: log(0) elkerülése
    h_t   = yt**2  # valódi variancia proxy: GK_RV²
    h_p   = yp**2  # becsült variancia: előrejelzés négyzetbe emelve
    qlike = np.mean(np.log(h_p + eps) + h_t / (h_p + eps))  # Q-LIKE: log(h²) + σ²/h² – Patton (2011), aszimmetrikusan bünteti az alábecsülést
    return {'Modell': name, 'Q-LIKE': round(qlike,6), 'RMSE': round(rmse,6),
            'MAE': round(mae,6), 'N': len(yt)}

def qlike_elementwise(y_true, y_pred):
    eps = 1e-10
    h_t = np.array(y_true)**2
    h_p = np.array(y_pred)**2
    return np.log(h_p + eps) + h_t / (h_p + eps)

def diebold_mariano_test(loss1, loss2, h=1):
    """Diebold-Mariano (1995) – HAC-robosztus"""
    d      = np.array(loss1) - np.array(loss2)  # veszteségkülönbség sorozat a DM-teszthez
    mask   = ~np.isnan(d)
    d      = d[mask]
    T      = len(d)
    if T < 10:
        return np.nan, np.nan
    mean_d = d.mean()
    gamma0 = np.sum((d - mean_d)**2) / T  # γ₀: a veszteségkülönbség varianciája
    lag    = max(0, h - 1)
    var_d  = gamma0
    if lag > 0:
        acov = sum(  # HAC (Newey-West) korrekció: autokorrelált veszteségek kezelése
            2 * (1 - l/(lag+1)) *
            np.sum((d[l:] - mean_d) * (d[:-l] - mean_d)) / T
            for l in range(1, lag+1)
        )
        var_d = gamma0 + acov
    dm_stat = mean_d / np.sqrt(max(var_d, 1e-10) / T)  # DM statisztika: d̄ / √(Var(d̄)), aszimptotikusan N(0,1)
    pval    = 2 * (1 - stats.norm.cdf(abs(dm_stat)))  # kétoldali p-érték normális eloszlás alapján
    return dm_stat, pval

print('Hibamérő függvények betöltve.')

In [ ]:
# =============================================================================
# KLASSZIKUS ROLLING WINDOW – EWMA, GARCH, GJR, HAR-RV
# =============================================================================
# HAR-RV (Corsi 2009): log(RV_t) = α + β_d·log(RV_{t-1})
#                                     + β_w·log(RV̄_{t-5})
#                                     + β_m·log(RV̄_{t-22}) + ε
# OLS rolling window – ugyanolyan struktúra mint EWMA/GARCH
# Visszatranszformálás: exp(ŷ) → évesített volatilitás
# =============================================================================
from sklearn.linear_model import LinearRegression

def run_classical_rolling(log_returns_full, analysis_log_ret, gk_rv,
                          prices_data_full,
                          rolling_window, lambda_ewma=0.94, refit_step=21):
    """
    Rolling window: EWMA, GARCH(1,1)-t, GJR-GARCH-t, HAR-RV (OLS)
    """
    start_idx = log_returns_full.index.get_loc(analysis_log_ret.index[0])  # az elemzési periódus kezdőindexe a teljes idősorban
    end_idx   = log_returns_full.index.get_loc(analysis_log_ret.index[-1]) + 1

    if start_idx < rolling_window:
        rolling_window = start_idx
        print(f'    ⚠ Ablak csökkentve: → {rolling_window} nap')

    refit_indices = set(range(start_idx, end_idx, refit_step))  # azok az időpontok ahol újra illesztjük a GARCH-ot (minden 21. nap)
    garch_params = gjr_params = None
    har_model    = None

    # HAR feature oszlopok – előre kiszámítva, shift(1)-gyel
    har_features = prices_data_full[['log_GK_d', 'log_GK_w', 'log_GK_m']].values  # HAR-RV feature-ök: előre kiszámított log(GK_daily) összetevők
    log_gk_target = np.log(gk_daily_full.clip(lower=1e-8).values)

    dates, ewma_f, garch_f, gjr_f, har_f = [], [], [], [], []

    for t in range(start_idx, end_idx):
        train   = log_returns_full.iloc[t - rolling_window : t]
        date    = log_returns_full.index[t]

        # EWMA
        ewma_val = train.ewm(alpha=1 - lambda_ewma).std().iloc[-1] * np.sqrt(252)  # EWMA std: exponenciálisan súlyozott mozgóátlag, λ=0.94

        if t in refit_indices:
            train100 = train * 100  # arch csomag %-os hozamot vár: numerikus stabilitás miatt szorzás 100-zal
            # GARCH(1,1)-t
            try:
                garch_params = arch_model(
                    train100, vol='Garch', p=1, q=1, dist='t'
                ).fit(disp='off', show_warning=False).params
            except Exception:
                pass
            # GJR-GARCH-t
            try:
                gjr_params = arch_model(
                    train100, vol='Garch', p=1, o=1, q=1, dist='t'  # GJR-GARCH: o=1 az aszimmetrikus leverage-tag – Glosten et al. (1993)
                ).fit(disp='off', show_warning=False).params
            except Exception:
                pass
            # HAR-RV OLS – log(GK_daily) ~ log_GK_d + log_GK_w + log_GK_m
            # Training ablak: t-rolling_window .. t-1
            X_har = har_features[t - rolling_window : t]
            y_har = log_gk_target[t - rolling_window : t]
            mask  = ~(np.isnan(X_har).any(axis=1) | np.isnan(y_har))
            if mask.sum() > 30:  # legalább 30 érvényes megfigyelés kell az OLS stabil becsléséhez
                har_model = LinearRegression().fit(X_har[mask], y_har[mask])

        def _forecast_garch(params, vol='Garch', o=0):
            if params is None:
                return np.nan
            try:
                m = arch_model(train * 100, vol=vol, p=1, o=o, q=1, dist='t').fix(params)
                v = m.forecast(horizon=1).variance.values[-1, 0]  # egy napra előre kondicionális variancia előrejelzés
                return np.sqrt(v) / 100 * np.sqrt(252)  # visszaskálázás: %-ból arányba (/100), napi→éves (×√252)
            except Exception:
                return np.nan

        # HAR-RV előrejelzés: t. nap feature-jei (shift(1)-gyel már leakage-mentes)
        if har_model is not None and not np.isnan(har_features[t]).any():
            har_pred = float(np.exp(har_model.predict(har_features[t].reshape(1, -1))[0]))  # exp visszatranszformáció: HAR-RV log-skálán becsül, de vol-skálán kell az eredmény
        else:
            har_pred = np.nan

        dates.append(date)
        ewma_f.append(ewma_val)
        garch_f.append(_forecast_garch(garch_params, 'Garch', 0))
        gjr_f.append(  _forecast_garch(gjr_params,   'Garch', 1))
        har_f.append(har_pred)

    res = pd.DataFrame(
        {'ewma': ewma_f, 'garch': garch_f, 'gjr': gjr_f, 'har': har_f},
        index=dates
    )
    res['GK_RV'] = gk_rv.reindex(dates).values
    return res

print('Klasszikus rolling window függvény betöltve (EWMA, GARCH, GJR, HAR-RV).')


In [ ]:
# =============================================================================
# LSTM ROLLING WINDOW – v13 (pontosan v8 logika)
# =============================================================================
# Input:   GK_daily múltja, MinMaxScaler(0,1) – v8 eredeti
# Target:  GK_daily (nem log-transzformált) – v8 eredeti
# Skálázás: MinMaxScaler(0,1) – v8 eredeti
# Nincs torch.compile, nincs grad clipping, nincs log/exp – tiszta v8
# =============================================================================

class LSTMVolModel(nn.Module):
    """Kétrétegű LSTM – Kim & Won (2018), Bucci (2020)."""
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(  # PyTorch beépített LSTM réteg; batch_first=True: (batch, seq, feature) bemenet
            input_size, hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0  # PyTorch figyelmeztetés elkerülése: 1 rétegnél a dropout paraméter hatástalan
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)  # kimeneti lineáris réteg: LSTM rejtett állapotából egyetlen volatilitás-érték

    def forward(self, x):
        out, _ = self.lstm(x)
        out     = self.dropout(out[:, -1, :])  # csak az utolsó időlépés rejtett állapota megy a fc rétegbe
        return self.fc(out).squeeze(-1)


def make_sequences(rv_series, lookback):
    """
    X[i] = GK_daily[i-lookback .. i-1]  (múlt)
    y[i] = GK_daily[i]                  (target)
    """
    X, y = [], []
    for i in range(lookback, len(rv_series)):  # csúszóablakos szekvencia-képzés: X[i]=rv[i-lookback:i], y[i]=rv[i]
        X.append(rv_series[i-lookback : i])
        y.append(rv_series[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def train_lstm(model, X_tr, y_tr, X_val, y_val,
               epochs=50, batch_size=32, patience=5, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # Adam optimalizáló: adaptív tanulási ráta, lr=0.001
    criterion = nn.MSELoss()  # MSE veszteségfüggvény a volatilitás-előrejelzéshez

    X_tr_t  = torch.tensor(X_tr).unsqueeze(-1).to(DEVICE)
    y_tr_t  = torch.tensor(y_tr).to(DEVICE)
    loader   = DataLoader(TensorDataset(X_tr_t, y_tr_t),  # mini-batch betöltő; shuffle=False: idősor-sorrendet meg kell tartani
                          batch_size=batch_size, shuffle=False)

    has_val = len(X_val) > 0
    if has_val:
        X_val_t = torch.tensor(X_val).unsqueeze(-1).to(DEVICE)
        y_val_t = torch.tensor(y_val).to(DEVICE)

    best_val_loss, best_state, wait = float('inf'), None, 0

    model.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

        if has_val:
            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(X_val_t), y_val_t).item()
            model.train()
            if val_loss < best_val_loss:  # korai leállás: legjobb validációs loss állapot mentése
                best_val_loss = val_loss
                best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1  # türelmi számláló: ha patience epoch óta nem javult, leállítás
                if wait >= patience:  # korai leállás aktiválása – overfitting megelőzése
                    break

    if has_val and best_state:
        model.load_state_dict(best_state)  # visszaállítás a legjobb checkpointra, nem a legutolsóra
    return model


def run_lstm_rolling(log_returns_full, gk_daily_full, analysis_log_ret,
                     rolling_window, lookback=60,
                     units=64, num_layers=2, dropout=0.1,
                     epochs=50, patience=5, batch_size=32, refit_step=5):
    """LSTM rolling window – v8 eredeti logika."""
    gk_daily_s = gk_daily_full.reindex(log_returns_full.index)

    start_idx = log_returns_full.index.get_loc(analysis_log_ret.index[0])
    end_idx   = log_returns_full.index.get_loc(analysis_log_ret.index[-1]) + 1

    if start_idx < rolling_window:
        rolling_window = start_idx

    refit_indices = set(range(start_idx, end_idx, refit_step))
    model         = None
    scaler        = MinMaxScaler(feature_range=(0, 1))
    dates, lstm_f = [], []

    for t in range(start_idx, end_idx):
        date = log_returns_full.index[t]
        dates.append(date)

        train_rv = gk_daily_s.iloc[t - rolling_window : t].values

        if np.isnan(train_rv).mean() > 0.3:
            lstm_f.append(np.nan)
            continue

        if t in refit_indices or model is None:
            train_rv_c = pd.Series(train_rv).interpolate().fillna(method='bfill').values  # hiányzó GK_daily értékek lineáris interpolációval pótolva
            scaled     = scaler.fit_transform(  # MinMaxScaler [0,1] skálázás – csak a training ablakra fit!
                train_rv_c.reshape(-1, 1)).flatten()

            X_all, y_all = make_sequences(scaled, lookback)

            if len(X_all) < lookback + 5:
                lstm_f.append(np.nan)
                continue

            split        = max(1, int(len(X_all) * 0.85))  # 85/15 train-validáció split az early stopping-hoz
            X_tr, y_tr   = X_all[:split], y_all[:split]
            X_val, y_val = X_all[split:], y_all[split:]

            torch.manual_seed(42)
            model = LSTMVolModel(
                hidden_size=units,
                num_layers=num_layers,
                dropout=dropout
            ).to(DEVICE)
            model = train_lstm(model, X_tr, y_tr, X_val, y_val,
                               epochs=epochs, batch_size=batch_size,
                               patience=patience)

        # Előrejelzés
        last_rv = gk_daily_s.iloc[t - lookback : t].values
        if np.isnan(last_rv).any():
            last_rv = pd.Series(last_rv).interpolate().fillna(method='bfill').values
        last_scaled = scaler.transform(last_rv.reshape(-1, 1)).flatten()  # CSAK transform, nem fit: a training scaler paramétereit használja

        inp = torch.tensor(last_scaled, dtype=torch.float32
                           ).unsqueeze(0).unsqueeze(-1).to(DEVICE)
        model.eval()
        with torch.no_grad():
            pred_scaled = model(inp).item()
        pred = scaler.inverse_transform([[pred_scaled]])[0, 0]  # visszaskálázás az eredeti volatilitás-tartományba
        lstm_f.append(max(pred, 0.001))  # negatív volatilitás fizikailag lehetetlen: alsó korlát 0.1%

    return pd.Series(lstm_f, index=dates, name='lstm')


print(f'PyTorch LSTM v13 betöltve. Device: {DEVICE}')
print(f'Input: GK_daily múltja (MinMaxScaler, lookback=60) – v8 eredeti logika')
print(f'Architektúra: 2 réteg, 64 neuron | Refit: 5 nap')


In [ ]:
# =============================================================================
# LSTM-GARCH HIBRID ROLLING WINDOW – v14 (Kim & Won 2018)
# =============================================================================
# Megközelítés: a klasszikus modellek előrejelzései mint LSTM input feature-ök
#
# Input (leakage-mentes, lookback=60 napos ablak):
#   X[t] = [GK_daily[t-lookback:t],     ← napi RV sorozat
#            ewma[t-lookback:t],          ← EWMA vol előrejelzés
#            garch[t-lookback:t],         ← GARCH(1,1) vol előrejelzés
#            gjr[t-lookback:t]]           ← GJR-GARCH vol előrejelzés
#   y[t] = GK_daily[t]                   ← target (v8 logika)
#
# Fontos: ewma/garch/gjr értékek a results_all-ból jönnek (már kiszámítva),
#         NEM fut újra GARCH illesztés – ez teszi gyorssá.
# Irodalom: Kim & Won (2018) Expert Systems with Applications 103:25-37
# =============================================================================

class LSTMHybridModel(nn.Module):  # Kim & Won (2018) hibrid architektúra: input_size=4 (GK+EWMA+GARCH+GJR)
    """Kétrétegű LSTM hibrid – Kim & Won (2018). input_size=4."""
    def __init__(self, input_size=4, hidden_size=64, num_layers=2, dropout=0.1):  # hibrid modell 4 bemeneti csatornával indul
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out     = self.dropout(out[:, -1, :])
        return self.fc(out).squeeze(-1)


def make_sequences_multi(feature_matrix, target_series, lookback):  # multivariate szekvencia: minden időlépéshez 4 feature-vektor
    """
    Multivariate szekvenciák.
    X[i]: feature_matrix[i-lookback:i]  shape: (lookback, n_features)
    y[i]: target_series[i]
    """
    X, y = [], []
    for i in range(lookback, len(target_series)):
        X.append(feature_matrix[i-lookback : i])
        y.append(target_series[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def train_lstm_multi(model, X_tr, y_tr, X_val, y_val,
                     epochs=50, batch_size=32, patience=5, lr=1e-3):
    """train_lstm multivariate – X mar (N, lookback, n_feat), nincs unsqueeze."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    X_tr_t  = torch.tensor(X_tr).to(DEVICE)
    y_tr_t  = torch.tensor(y_tr).to(DEVICE)
    loader   = DataLoader(TensorDataset(X_tr_t, y_tr_t),
                          batch_size=batch_size, shuffle=False)

    has_val = len(X_val) > 0
    if has_val:
        X_val_t = torch.tensor(X_val).to(DEVICE)
        y_val_t = torch.tensor(y_val).to(DEVICE)

    best_val_loss, best_state, wait = float('inf'), None, 0

    model.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

        if has_val:
            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(X_val_t), y_val_t).item()
            model.train()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break

    if has_val and best_state:
        model.load_state_dict(best_state)
    return model


def run_lstm_hybrid_rolling(log_returns_full, gk_daily_full, classical_res,
                             analysis_log_ret, rolling_window, lookback=60,
                             units=64, num_layers=2, dropout=0.1,
                             epochs=50, patience=5, batch_size=32,
                             refit_step=21):
    """
    LSTM-GARCH hibrid rolling window – Kim & Won (2018).

    classical_res: results_all[pid][rw] DataFrame, tartalmazza:
                   ewma, garch, gjr oszlopokat (már kiszámítva)
    Feature mátrix: GK_daily + ewma + garch + gjr (4 feature)
    Target: GK_daily (v8 logika, MinMaxScaler visszatranszformálás)
    """
    gk_daily_s = gk_daily_full.reindex(log_returns_full.index)

    # Klasszikus előrejelzések a teljes időszakra (már kiszámítva)
    ewma_s  = classical_res['ewma'].reindex(log_returns_full.index)
    garch_s = classical_res['garch'].reindex(log_returns_full.index)
    gjr_s   = classical_res['gjr'].reindex(log_returns_full.index)

    start_idx = log_returns_full.index.get_loc(analysis_log_ret.index[0])
    end_idx   = log_returns_full.index.get_loc(analysis_log_ret.index[-1]) + 1

    if start_idx < rolling_window:
        rolling_window = start_idx

    refit_indices = set(range(start_idx, end_idx, refit_step))
    model         = None
    scalers       = [MinMaxScaler(feature_range=(0, 1)) for _ in range(4)]
    dates, hybrid_f = [], []

    for t in range(start_idx, end_idx):
        date = log_returns_full.index[t]
        dates.append(date)

        # Training ablak – 4 feature
        gk_win   = gk_daily_s.iloc[t - rolling_window : t].values
        ewma_win  = ewma_s.iloc[t - rolling_window : t].values
        garch_win = garch_s.iloc[t - rolling_window : t].values
        gjr_win   = gjr_s.iloc[t - rolling_window : t].values

        if np.isnan(gk_win).mean() > 0.3:
            hybrid_f.append(np.nan)
            continue

        # NaN interpoláció
        def clean(arr):  # NaN kezelés: interpoláció → ffill → bfill → 0.0 végső fallback
            s = pd.Series(arr).interpolate().ffill().bfill().values
            return np.where(np.isnan(s), 0.0, s)

        gk_c    = clean(gk_win)
        ewma_c  = clean(ewma_win)
        garch_c = clean(garch_win)
        gjr_c   = clean(gjr_win)

        raw_features = np.column_stack([gk_c, ewma_c, garch_c, gjr_c])  # 4 feature oszlop: GK_daily, EWMA, GARCH, GJR – mind éves vol skálán

        if t in refit_indices or model is None:
            # Scalerek fit a training ablakra
            scaled_features = np.column_stack([
                scalers[i].fit_transform(raw_features[:, i].reshape(-1, 1)).flatten()  # minden featuret külön MinMaxScaler skáláz – eltérő tartományok miatt
                for i in range(4)
            ])
            target_scaled = scalers[0].transform(gk_c.reshape(-1, 1)).flatten()  # target skálázás a GK_daily scalerével (scalers[0]) – konzisztens visszatranszformációhoz

            X_all, y_all = make_sequences_multi(scaled_features, target_scaled, lookback)

            if len(X_all) < lookback + 5:
                hybrid_f.append(np.nan)
                continue

            split        = max(1, int(len(X_all) * 0.85))
            X_tr, y_tr   = X_all[:split], y_all[:split]
            X_val, y_val = X_all[split:], y_all[split:]

            torch.manual_seed(42)
            model = LSTMHybridModel(
                input_size=4,  # hibrid modell 4 bemeneti csatornával indul
                hidden_size=units,
                num_layers=num_layers,
                dropout=dropout
            ).to(DEVICE)
            model = train_lstm_multi(model, X_tr, y_tr, X_val, y_val,
                                     epochs=epochs, batch_size=batch_size,
                                     patience=patience)

        # Előrejelzés: utolsó lookback nap 4 feature-je
        gk_last    = clean(gk_daily_s.iloc[t - lookback : t].values)
        ewma_last  = clean(ewma_s.iloc[t - lookback : t].values)
        garch_last = clean(garch_s.iloc[t - lookback : t].values)
        gjr_last   = clean(gjr_s.iloc[t - lookback : t].values)

        last_raw = np.column_stack([gk_last, ewma_last, garch_last, gjr_last])
        last_scaled = np.column_stack([
            scalers[i].transform(last_raw[:, i].reshape(-1, 1)).flatten()
            for i in range(4)
        ])

        inp = torch.tensor(last_scaled[np.newaxis], dtype=torch.float32).to(DEVICE)
        model.eval()
        with torch.no_grad():
            pred_scaled = model(inp).item()
        pred = scalers[0].inverse_transform([[pred_scaled]])[0, 0]  # visszaskálázás a GK_daily scaler inverzzével
        hybrid_f.append(max(float(pred), 0.001))

    return pd.Series(hybrid_f, index=dates, name='hybrid')


print(f'LSTM-GARCH Hibrid v14 betöltve. Device: {DEVICE}')
print(f'Input: GK_daily + EWMA + GARCH + GJR (4 feature, előre kiszámítva)')
print(f'Irodalom: Kim & Won (2018) Expert Systems with Applications 103:25-37')


In [ ]:
# =============================================================================
# KLASSZIKUS MODELLEK FUTTATÁSA – 3 periódus (EWMA, GARCH, GJR, HAR-RV)
# =============================================================================
results_all = {pid: {} for pid in PERIODS}

for pid, pd_ in period_data.items():  # iteráció a 3 perióduson: Pre-COVID, COVID-sokk, Kamatemelés
    print(f'\n{"="*65}')
    print(f'Periódus {pid}: {pd_["name"]} ({pd_["start"]} – {pd_["end"]})')
    print(f'{"="*65}')
    for rw in ROLLING_WINDOWS:  # iteráció a rolling window méreteken (jelen esetben csak 1000)
        print(f'  Klasszikus RW={rw}...', end=' ', flush=True)
        res = run_classical_rolling(  # klasszikus modellek (EWMA, GARCH, GJR, HAR-RV) futtatása egy periódusra
            log_returns_full = log_returns_full,
            analysis_log_ret = pd_['log_ret'],
            gk_rv            = pd_['gk_rv'],
            prices_data_full = prices_data_full,
            rolling_window   = rw,
            lambda_ewma      = LAMBDA_EWMA,
            refit_step       = REFIT_STEP,
        )
        results_all[pid][rw] = res  # eredmény tárolása: dict[periódus_id][rolling_window] → DataFrame
        n_valid = res.dropna().shape[0]
        print(f'✓ ({n_valid} érvényes nap)')

print('\n✓ Klasszikus modellek kész (EWMA, GARCH, GJR, HAR-RV).')


In [ ]:
# =============================================================================
# LSTM FUTTATÁSA – v13
# =============================================================================
lstm_results = {pid: {} for pid in PERIODS}

for pid, pd_ in period_data.items():
    print(f'\n{"="*65}')
    print(f'LSTM v13 – Periódus {pid}: {pd_["name"]}')
    print(f'{"="*65}')
    for rw in ROLLING_WINDOWS:
        print(f'  LSTM RW={rw}...', flush=True)
        lstm_pred = run_lstm_rolling(
            log_returns_full = log_returns_full,
            gk_daily_full    = gk_daily_full,
            analysis_log_ret = pd_['log_ret'],
            rolling_window   = rw,
            lookback         = LSTM_LOOKBACK,
            units            = LSTM_UNITS,
            num_layers       = LSTM_LAYERS,
            dropout          = LSTM_DROPOUT,
            epochs           = LSTM_EPOCHS,
            patience         = LSTM_PATIENCE,
            batch_size       = LSTM_BATCH,
            refit_step       = LSTM_REFIT,
        )
        results_all[pid][rw]['lstm'] = lstm_pred.reindex(results_all[pid][rw].index).values
        lstm_results[pid][rw] = lstm_pred
        n_valid = lstm_pred.dropna().shape[0]
        print(f'  ✓ LSTM RW={rw} kész ({n_valid} érvényes nap)')

print('\n✓ Összes LSTM futtatás kész.')


In [ ]:
# =============================================================================
# LSTM-GARCH HIBRID FUTTATÁSA – 3 periódus
# =============================================================================
for pid, pd_ in period_data.items():
    print(f'\n{"="*65}')
    print(f'LSTM-GARCH Hibrid – Periódus {pid}: {pd_["name"]}')
    print(f'{"="*65}')
    for rw in ROLLING_WINDOWS:
        print(f'  Hibrid RW={rw}...', flush=True)
        hybrid_pred = run_lstm_hybrid_rolling(
            log_returns_full = log_returns_full,
            gk_daily_full    = gk_daily_full,
            classical_res    = results_all[pid][rw],
            analysis_log_ret = pd_['log_ret'],
            rolling_window   = rw,
            lookback         = LSTM_LOOKBACK,
            units            = LSTM_UNITS,
            num_layers       = LSTM_LAYERS,
            dropout          = LSTM_DROPOUT,
            epochs           = LSTM_EPOCHS,
            patience         = LSTM_PATIENCE,
            batch_size       = LSTM_BATCH,
            refit_step       = LSTM_REFIT,
        )
        results_all[pid][rw]['hybrid'] = hybrid_pred.reindex(
            results_all[pid][rw].index).values
        n_valid = hybrid_pred.dropna().shape[0]
        print(f'  ✓ Hibrid RW={rw} kész ({n_valid} érvényes nap)')

print('\n✓ LSTM-GARCH hibrid futtatás kész.')


In [ ]:
# =============================================================================
# HIBAMÉRŐK – GK_RV ground truth
# =============================================================================
MODEL_COLS = [  # az összehasonlítandó modellek belső neve és megjelenített neve
    ('ewma',  'EWMA'),
    ('garch', 'GARCH(1,1)'),
    ('gjr',   'GJR-GARCH'),
    ('har',   'HAR-RV'),
    ('lstm',  'LSTM'),
    ('hybrid', 'LSTM-GARCH'),
]

rows_metrics = []
for pid, pd_ in period_data.items():
    for rw in ROLLING_WINDOWS:
        res = results_all[pid][rw].dropna()
        gt  = res['GK_RV']  # ground truth: Garman-Klass Realized Volatility 21 napos rolling
        for col, name in MODEL_COLS:
            if col in res.columns:
                m = calc_metrics(gt, res[col], name)
                m['Periódus'] = f'[{pid}] {pd_["name"]}'
                m['RW']       = rw
                rows_metrics.append(m)

metrics_df    = pd.DataFrame(rows_metrics)
metrics_pivot = metrics_df.pivot_table(  # pivot tábla: sorok=periódus, oszlopok=modell, értékek=Q-LIKE/RMSE/MAE
    index=['Periódus', 'RW'],
    columns='Modell',
    values=['Q-LIKE', 'RMSE', 'MAE']
)

print('\n' + '='*80)
print('HIBAMÉRŐK – Ground truth: GK_RV (Garman-Klass Realized Volatility)')
print('Értelmezés: Q-LIKE alapján rangsorolj (kisebb = jobb) – Patton (2011)')
print('='*80)
from IPython.display import display
display(metrics_pivot.round(6))
metrics_df.to_csv(f'{FIG_DIR}/metrics_GK_RV_{safe_ticker}.csv', index=False)
print(f'✓ Mentve: figures/metrics_GK_RV_{safe_ticker}.csv')


In [ ]:
# =============================================================================
# DIEBOLD-MARIANO TESZTEK + HOLM-BONFERRONI KORREKCIÓ
# Romano & Wolf (2005) multiple testing
# =============================================================================
DM_PAIRS = [  # 15 modell-pár: minden modell összehasonlítva minden másikkal
    ('EWMA',       'ewma',  'GARCH(1,1)', 'garch'),
    ('EWMA',       'ewma',  'GJR-GARCH',  'gjr'),
    ('EWMA',       'ewma',  'HAR-RV',     'har'),
    ('EWMA',       'ewma',  'LSTM',       'lstm'),
    ('GARCH(1,1)', 'garch', 'GJR-GARCH',  'gjr'),
    ('GARCH(1,1)', 'garch', 'HAR-RV',     'har'),
    ('GARCH(1,1)', 'garch', 'LSTM',       'lstm'),
    ('GJR-GARCH',  'gjr',   'HAR-RV',     'har'),
    ('GJR-GARCH',  'gjr',   'LSTM',       'lstm'),
    ('HAR-RV',     'har',   'LSTM',       'lstm'),
    ('EWMA',       'ewma',  'LSTM-GARCH', 'hybrid'),
    ('GARCH(1,1)', 'garch', 'LSTM-GARCH', 'hybrid'),
    ('GJR-GARCH',  'gjr',   'LSTM-GARCH', 'hybrid'),
    ('HAR-RV',     'har',   'LSTM-GARCH', 'hybrid'),
    ('LSTM',       'lstm',  'LSTM-GARCH', 'hybrid'),
]

def sig_label(p):
    if pd.isna(p): return 'n/a'
    return '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else 'n.s.'))

rows_dm = []
for pid, pd_ in period_data.items():
    for rw in ROLLING_WINDOWS:
        res = results_all[pid][rw].dropna()
        gt  = res['GK_RV']
        for n1, c1, n2, c2 in DM_PAIRS:
            if c1 not in res.columns or c2 not in res.columns:
                continue
            l1 = qlike_elementwise(gt, res[c1])  # Q-LIKE veszteség sorozat az első modellhez (elemenként)
            l2 = qlike_elementwise(gt, res[c2])  # Q-LIKE veszteség sorozat a második modellhez
            dm, pval = diebold_mariano_test(l1, l2, h=1)
            better = n1 if dm > 0 else n2  # DM>0: l1 átlagosan nagyobb → l2 (2. modell) jobb
            rows_dm.append({
                'Periódus': f'[{pid}] {pd_["name"]}',
                'RW': rw,
                'Modell 1': n1, 'Modell 2': n2,
                'DM stat': round(dm, 4),
                'p_raw': round(pval, 4),
                'Szign. (raw)': sig_label(pval),
                'Jobb': better,
            })

dm_df = pd.DataFrame(rows_dm)

# Holm-Bonferroni korrekció
for pid in PERIODS:
    for rw in ROLLING_WINDOWS:
        mask = (dm_df['Periódus'] == f'[{pid}] {period_data[pid]["name"]}') & (dm_df['RW'] == rw)
        if mask.sum() == 0:
            continue
        pvals = dm_df.loc[mask, 'p_raw'].values
        _, p_corr, _, _ = multipletests(pvals, method='holm')  # Holm-Bonferroni korrekció: FWER kontroll 15 egyidejű tesztnél
        dm_df.loc[mask, 'p_korr (Holm)'] = p_corr.round(4)
        dm_df.loc[mask, 'Szign. (Holm)'] = [sig_label(p) for p in p_corr]

print('\n' + '='*80)
print('DIEBOLD-MARIANO TESZTEK – Ground truth: GK_RV')
print('Holm-Bonferroni korrekció (Romano & Wolf, 2005)')
print('='*80)
print('  *** p<0.01  ** p<0.05  * p<0.10  n.s. nem szignifikáns')
print()
from IPython.display import display
display(dm_df)

changed = (dm_df['Szign. (raw)'] != dm_df['Szign. (Holm)']).sum()
print(f'\n✓ Mentve: figures/dm_tests_{safe_ticker}.csv')
print(f'  Korrekció hatása: {changed} teszt szignifikancia-szintje változott.')
dm_df.to_csv(f'{FIG_DIR}/dm_tests_{safe_ticker}.csv', index=False)


In [ ]:
# 7.1 Rolling window idősorok – periódusokként, 3×1 vertikális grid
MODEL_PLOT = [
    ('ewma',  f'EWMA λ={LAMBDA_EWMA}', '#58DB94'),
    ('garch', 'GARCH(1,1)',             '#E34A17'),
    ('gjr',   'GJR-GARCH',              '#18DB30'),
    ('har',   'HAR-RV',                 '#E3628D'),
    ('lstm',   'LSTM',                   '#174DE3'),
    ('hybrid', 'LSTM-GARCH Hibrid',      '#E39717'),
]

for rw in ROLLING_WINDOWS:
    fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=False)
    for i, (pid, pd_) in enumerate(period_data.items()):
        ax  = axes[i]
        res = results_all[pid][rw]
        ax.plot(res.index, res['GK_RV'], color='black', lw=2.0,  # fekete vastag vonal: ground truth GK_RV referencia
                label='GK_RV (ground truth)', zorder=5)
        for col, label, color in MODEL_PLOT:
            if col in res.columns and not res[col].isna().all():
                ax.plot(res.index, res[col], color=color, lw=1.2,  # modellek előrejelzései különböző színekkel
                        alpha=0.8, label=label)
        ax.set_title(pd_['name'], fontsize=12, fontweight='bold', pad=8)
        ax.set_ylabel('Évesített volatilitás', fontsize=10)
        ax.legend(fontsize=9, loc='upper right')
        ax.tick_params(axis='x', rotation=20)
        ax.grid(True, alpha=0.3)
        if i == 2:
            ax.set_xlabel('Dátum', fontsize=10)
    plt.suptitle(f'{DISPLAY_NAME} – Volatilitás-előrejelzések (RW={rw} nap)',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    fname = f'{FIG_DIR}/05_rolling_GK_RV_RW{rw}_{safe_ticker}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Mentve: {fname}')


In [ ]:
# 7.2 Q-LIKE és RMSE barplot – külön ábrák
colors_bar = {
    'ewma':  '#e74c3c',
    'garch': '#8e44ad',
    'gjr':   '#27ae60',
    'har':   '#2980b9',
    'lstm':   '#f39c12',
    'hybrid': '#1abc9c',
}

for rw in ROLLING_WINDOWS:
    for metric, ylabel, title_suffix, fname_tag in [
        ('Q-LIKE', 'Q-LIKE (kisebb = jobb)',  'Q-LIKE',  'qlike'),
        ('RMSE',   'RMSE (kisebb = jobb)',     'RMSE',    'rmse'),
    ]:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
        for i, (pid, pd_) in enumerate(period_data.items()):
            ax  = axes[i]
            res = results_all[pid][rw].dropna()
            gt  = res['GK_RV']
            vals, names, bar_colors = [], [], []
            for col, label in MODEL_COLS:
                if col in res.columns:
                    m = calc_metrics(gt, res[col], label)
                    vals.append(m[metric])
                    names.append(label)
                    bar_colors.append(colors_bar[col])
            bars = ax.bar(names, vals, color=bar_colors,
                          edgecolor='white', linewidth=0.5)
            ax.set_title(pd_['name'], fontsize=10, fontweight='bold')
            ax.set_ylabel(ylabel, fontsize=9)
            ax.tick_params(axis='x', rotation=20, labelsize=8)
            ax.grid(True, alpha=0.3, axis='y')
            best_idx = int(np.argmin(vals))
            bars[best_idx].set_edgecolor('black')
            bars[best_idx].set_linewidth(2.5)
        plt.suptitle(
            f'{DISPLAY_NAME} – {title_suffix} (RW={rw}) | kisebb = jobb',
            fontsize=12, fontweight='bold'
        )
        plt.tight_layout()
        fname = f'{FIG_DIR}/06_{fname_tag}_barplot_RW{rw}_{safe_ticker}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'✓ Mentve: {fname}')


In [ ]:
# 7.3 DM heatmap – Holm-korrigált szignifikancia (5 modell)
for rw in ROLLING_WINDOWS:
    dm_rw        = dm_df[dm_df['RW'] == rw].copy()
    periods_list = [pd_['name'] for pd_ in period_data.values()]
    model_list   = ['EWMA', 'GARCH(1,1)', 'GJR-GARCH', 'HAR-RV', 'LSTM', 'LSTM-GARCH']
    n_m          = len(model_list)

    fig, axes = plt.subplots(1, len(periods_list),
                             figsize=(5.5 * len(periods_list), 5.5))
    if len(periods_list) == 1:
        axes = [axes]

    for ax, (pid, pd_) in zip(axes, period_data.items()):
        dm_p = dm_rw[dm_rw['Periódus'] == f'[{pid}] {pd_["name"]}']
        mat  = np.full((n_m, n_m), np.nan)
        for _, row in dm_p.iterrows():
            if row['Modell 1'] in model_list and row['Modell 2'] in model_list:
                i1 = model_list.index(row['Modell 1'])
                i2 = model_list.index(row['Modell 2'])
                p  = row.get('p_korr (Holm)', row['p_raw'])
                if not pd.isna(p):
                    mat[i1, i2] = p
                    mat[i2, i1] = p

        annot = np.full((n_m, n_m), '', dtype=object)
        for r in range(n_m):
            for c in range(n_m):
                if not np.isnan(mat[r, c]):
                    annot[r, c] = sig_label(mat[r, c])

        sns.heatmap(mat, ax=ax, annot=annot, fmt='',
                    xticklabels=model_list, yticklabels=model_list,
                    cmap='RdYlGn_r', vmin=0, vmax=0.1,
                    cbar_kws={'label': 'p-érték (Holm)'},
                    linewidths=0.5)
        ax.set_title(pd_['name'], fontsize=10, fontweight='bold')
        ax.tick_params(axis='x', rotation=20)

    plt.suptitle(f'{DISPLAY_NAME} – DM-teszt heatmap (Holm, RW={rw})',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    fname = f'{FIG_DIR}/07_dm_heatmap_RW{rw}_{safe_ticker}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Mentve: {fname}')
